In [1]:
%matplotlib inline


Training a Classifier
=====================

This is it. You have seen how to define neural networks, compute loss and make
updates to the weights of the network.

Now you might be thinking,

What about data?
----------------

Generally, when you have to deal with image, text, audio or video data,
you can use standard python packages that load data into a numpy array.
Then you can convert this array into a ``torch.*Tensor``.

-  For images, packages such as Pillow, OpenCV are useful
-  For audio, packages such as scipy and librosa
-  For text, either raw Python or Cython based loading, or NLTK and
   SpaCy are useful

Specifically for vision, we have created a package called
``torchvision``, that has data loaders for common datasets such as
Imagenet, CIFAR10, MNIST, etc. and data transformers for images, viz.,
``torchvision.datasets`` and ``torch.utils.data.DataLoader``.

This provides a huge convenience and avoids writing boilerplate code.

For this tutorial, we will use the CIFAR10 dataset.
It has the classes: ‘airplane’, ‘automobile’, ‘bird’, ‘cat’, ‘deer’,
‘dog’, ‘frog’, ‘horse’, ‘ship’, ‘truck’. The images in CIFAR-10 are of
size 3x32x32, i.e. 3-channel color images of 32x32 pixels in size.

.. figure:: /_static/img/cifar10.png
   :alt: cifar10

   cifar10


Training an image classifier
----------------------------

We will do the following steps in order:

1. Load and normalizing the CIFAR10 training and test datasets using
   ``torchvision``
2. Define a Convolution Neural Network
3. Define a loss function
4. Train the network on the training data
5. Test the network on the test data

1. Loading and normalizing CIFAR10
^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

Using ``torchvision``, it’s extremely easy to load CIFAR10.



In [2]:
import torch
import torchvision
import torchvision.transforms as transforms
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(device)


cuda:0


The output of torchvision datasets are PILImage images of range [0, 1].
We transform them to Tensors of normalized range [-1, 1].



In [3]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=4,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=4,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

100%|██████████| 170M/170M [00:19<00:00, 8.89MB/s]


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self):
        super().__init__()

        # ===== Block 1 =====
        self.conv1 = nn.Conv2d(3, 8, 3, padding=1)
        self.bn1   = nn.BatchNorm2d(8)

        self.conv2 = nn.Conv2d(3, 8, 5, padding=2)
        self.bn2   = nn.BatchNorm2d(8)

        self.pool1 = nn.MaxPool2d(2, 2)

        self.conv3 = nn.Conv2d(16, 16, 3, stride=1, padding=1)
        self.bn3   = nn.BatchNorm2d(16)

        self.conv4 = nn.Conv2d(16, 32, 5, padding=2)
        self.bn4   = nn.BatchNorm2d(32)

        self.conv5 = nn.Conv2d(16, 16, 3, padding=1)
        self.bn5   = nn.BatchNorm2d(16)

        self.conv6 = nn.Conv2d(16, 32, 5, padding=2)
        self.bn6   = nn.BatchNorm2d(32)

        # ===== Block 2 =====
        self.conv7 = nn.Conv2d(32, 32, 3, padding=1)
        self.bn7   = nn.BatchNorm2d(32)

        self.conv8 = nn.Conv2d(32, 32, 5, padding=2)
        self.bn8   = nn.BatchNorm2d(32)

        self.conv9 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn9   = nn.BatchNorm2d(64)

        self.conv10 = nn.Conv2d(64, 64, 5, padding=2)
        self.bn10   = nn.BatchNorm2d(64)

        self.conv11 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn11   = nn.BatchNorm2d(64)

        self.conv12 = nn.Conv2d(64, 64, 5, padding=2)
        self.bn12   = nn.BatchNorm2d(64)

        self.pool2 = nn.MaxPool2d(2, 2)

        # ===== Block 3 =====
        self.conv13 = nn.Conv2d(128, 128, 3, padding=1)
        self.bn13   = nn.BatchNorm2d(128)

        self.conv14 = nn.Conv2d(128, 256, 5, padding=2)
        self.bn14   = nn.BatchNorm2d(256)

        self.conv15 = nn.Conv2d(128, 128, 3, padding=1)
        self.bn15   = nn.BatchNorm2d(128)

        self.conv16 = nn.Conv2d(128, 256, 5, padding=2)
        self.bn16   = nn.BatchNorm2d(256)

        self.pool3 = nn.MaxPool2d(2, 2)

        # ===== Block 4 =====
        self.conv17 = nn.Conv2d(512, 258, 3, padding=1)
        self.bn17   = nn.BatchNorm2d(258)

        self.conv18 = nn.Conv2d(258, 258, 5, padding=2)
        self.bn18   = nn.BatchNorm2d(258)

        self.conv19 = nn.Conv2d(512, 258, 3, padding=1)
        self.bn19   = nn.BatchNorm2d(258)

        self.conv20 = nn.Conv2d(258, 258, 5, padding=2)
        self.bn20   = nn.BatchNorm2d(258)

        # ===== FC =====
        self.fc1 = nn.Linear(258 * 4 * 4, 512)
        self.fc2 = nn.Linear(512, 10)

    def forward(self, x):
        # ===== Block 1 =====
        y = F.relu(self.bn1(self.conv1(x)))
        z = F.relu(self.bn2(self.conv2(x)))
        x = torch.cat([y, z], dim=1)
        x = self.pool1(x)

        x1 = F.relu(self.bn3(self.conv3(x)))
        x1 = F.relu(self.bn4(self.conv4(x1)))

        x  = F.relu(self.bn5(self.conv5(x)))
        x  = F.relu(self.bn6(self.conv6(x)))
        x  = x + x1

        # ===== Block 2 =====
        y = F.relu(self.bn7(self.conv7(x)))
        z = F.relu(self.bn8(self.conv8(y)))
        x = y + z
        y = F.relu(self.bn9(self.conv9(x)))
        y = F.relu(self.bn10(self.conv10(y)))

        z = F.relu(self.bn11(self.conv11(x)))
        z = F.relu(self.bn12(self.conv12(z)))

        x = torch.cat([y, z], dim=1)
        x = self.pool2(x)

        # ===== Block 3 =====
        y = F.relu(self.bn13(self.conv13(x)))
        y = F.relu(self.bn14(self.conv14(y)))

        z = F.relu(self.bn15(self.conv15(x)))
        z = F.relu(self.bn16(self.conv16(z)))

        x = torch.cat([y, z], dim=1)
        x = self.pool3(x)

        # ===== Block 4 =====
        y = F.relu(self.bn17(self.conv17(x)))
        y = F.relu(self.bn18(self.conv18(y)))

        z = F.relu(self.bn19(self.conv19(x)))
        z = F.relu(self.bn20(self.conv20(z)))

        x = y + z

        # ===== FC =====
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x


net = Net()
net.to(device)

from torchsummary import summary
summary(net, (3, 32, 32))


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1            [-1, 8, 32, 32]             224
       BatchNorm2d-2            [-1, 8, 32, 32]              16
            Conv2d-3            [-1, 8, 32, 32]             608
       BatchNorm2d-4            [-1, 8, 32, 32]              16
         MaxPool2d-5           [-1, 16, 16, 16]               0
            Conv2d-6           [-1, 16, 16, 16]           2,320
       BatchNorm2d-7           [-1, 16, 16, 16]              32
            Conv2d-8           [-1, 32, 16, 16]          12,832
       BatchNorm2d-9           [-1, 32, 16, 16]              64
           Conv2d-10           [-1, 16, 16, 16]           2,320
      BatchNorm2d-11           [-1, 16, 16, 16]              32
           Conv2d-12           [-1, 32, 16, 16]          12,832
      BatchNorm2d-13           [-1, 32, 16, 16]              64
           Conv2d-14           [-1, 32,

In [5]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [6]:
for epoch in range(30):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)
        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 2000 == 1999:    # print every 2000 mini-batches
            print('[%d, %5d] loss: %.3f' %
                  (epoch + 1, i + 1, running_loss / 2000))
            running_loss = 0.0

print('Finished Training')

[1,  2000] loss: 1.953
[1,  4000] loss: 1.759
[1,  6000] loss: 1.662
[1,  8000] loss: 1.571
[1, 10000] loss: 1.500
[1, 12000] loss: 1.432
[2,  2000] loss: 1.311
[2,  4000] loss: 1.288
[2,  6000] loss: 1.261
[2,  8000] loss: 1.203
[2, 10000] loss: 1.164
[2, 12000] loss: 1.128
[3,  2000] loss: 1.062
[3,  4000] loss: 1.018
[3,  6000] loss: 1.026
[3,  8000] loss: 0.989
[3, 10000] loss: 0.969
[3, 12000] loss: 0.949
[4,  2000] loss: 0.856
[4,  4000] loss: 0.872
[4,  6000] loss: 0.861
[4,  8000] loss: 0.835
[4, 10000] loss: 0.815
[4, 12000] loss: 0.848
[5,  2000] loss: 0.743
[5,  4000] loss: 0.703
[5,  6000] loss: 0.716
[5,  8000] loss: 0.756
[5, 10000] loss: 0.724
[5, 12000] loss: 0.725
[6,  2000] loss: 0.596
[6,  4000] loss: 0.629
[6,  6000] loss: 0.650
[6,  8000] loss: 0.630
[6, 10000] loss: 0.639
[6, 12000] loss: 0.630
[7,  2000] loss: 0.530
[7,  4000] loss: 0.535
[7,  6000] loss: 0.555
[7,  8000] loss: 0.547
[7, 10000] loss: 0.545
[7, 12000] loss: 0.558
[8,  2000] loss: 0.437
[8,  4000] 

In [10]:
correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print('Accuracy of the network on the 10000 test images: %d %%' % (
    100 * correct / total))

Accuracy of the network on the 10000 test images: 79 %


That looks waaay better than chance, which is 10% accuracy (randomly picking
a class out of 10 classes).
Seems like the network learnt something.

Hmmm, what are the classes that performed well, and the classes that did
not perform well:



In [11]:
class_correct = list(0. for i in range(10))
class_total = list(0. for i in range(10))
with torch.no_grad():
    for data in testloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        c = (predicted == labels).squeeze()
        for i in range(4):
            label = labels[i]
            class_correct[label] += c[i].item()
            class_total[label] += 1


for i in range(10):
    print('Accuracy of %5s : %2d %%' % (
        classes[i], 100 * class_correct[i] / class_total[i]))

Accuracy of plane : 82 %
Accuracy of   car : 92 %
Accuracy of  bird : 74 %
Accuracy of   cat : 66 %
Accuracy of  deer : 75 %
Accuracy of   dog : 71 %
Accuracy of  frog : 82 %
Accuracy of horse : 77 %
Accuracy of  ship : 88 %
Accuracy of truck : 84 %


Okay, so what next?

How do we run these neural networks on the GPU?

Training on GPU
----------------
Just like how you transfer a Tensor on to the GPU, you transfer the neural
net onto the GPU.

Let's first define our device as the first visible cuda device if we have
CUDA available:



In [9]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Assume that we are on a CUDA machine, then this should print a CUDA device:

print(device)

cuda:0


The rest of this section assumes that `device` is a CUDA device.

Then these methods will recursively go over all modules and convert their
parameters and buffers to CUDA tensors:

.. code:: python

    net.to(device)


Remember that you will have to send the inputs and targets at every step
to the GPU too:

.. code:: python

        inputs, labels = inputs.to(device), labels.to(device)

Why dont I notice MASSIVE speedup compared to CPU? Because your network
is realllly small.

**Exercise:** Try increasing the width of your network (argument 2 of
the first ``nn.Conv2d``, and argument 1 of the second ``nn.Conv2d`` –
they need to be the same number), see what kind of speedup you get.

**Goals achieved**:

- Understanding PyTorch's Tensor library and neural networks at a high level.
- Train a small neural network to classify images

Training on multiple GPUs
-------------------------
If you want to see even more MASSIVE speedup using all of your GPUs,
please check out :doc:`data_parallel_tutorial`.

Where do I go next?
-------------------

-  :doc:`Train neural nets to play video games </intermediate/reinforcement_q_learning>`
-  `Train a state-of-the-art ResNet network on imagenet`_
-  `Train a face generator using Generative Adversarial Networks`_
-  `Train a word-level language model using Recurrent LSTM networks`_
-  `More examples`_
-  `More tutorials`_
-  `Discuss PyTorch on the Forums`_
-  `Chat with other users on Slack`_


